# 최종 네트워크를 위한 키워드 추출 및 분석 (Instagram Data)

이 노트북은 정제된 인스타그램 제품 데이터를 바탕으로 최종 네트워크 구성을 위한 키워드 추출 작업을 수행합니다.
이전 단계에서 생성된 `df_qual_checkpoint.pkl` 데이터를 기반으로 합니다.

## Step 1. 환경 설정 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import json
import ast
import pickle
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')

BASE_DIR = r'C:\Users\송정현\Documents\Projects\박재홍교수님세미나\Projects\20기\7eleven_npd_framework'
CHECKPOINT_PATH = os.path.join(BASE_DIR, 'eda', 'df_qual_checkpoint.pkl')

if os.path.exists(CHECKPOINT_PATH):
    df_qual = pd.read_pickle(CHECKPOINT_PATH)
    print(f"✅ 체크포인트 로드 완료: {len(df_qual)}개 제품")
else:
    print(f"❌ 체크포인트 파일을 찾을 수 없습니다: {CHECKPOINT_PATH}")

## Step 2. 키워드 빈도 분석 및 정제 대상 탐색

In [ ]:
from collections import Counter

all_kws = []
for kw_list in df_qual['확정키워드_최종']:
    if isinstance(kw_list, list):
        all_kws.extend(kw_list)

kw_freq = Counter(all_kws)
df_freq = pd.DataFrame(kw_freq.most_common(), columns=['키워드', '빈도'])
df_freq['등장_제품수'] = df_freq['키워드'].apply(
    lambda kw: df_qual['확정키워드_최종'].apply(
        lambda x: kw in x if isinstance(x, list) else False
    ).sum()
)

print(f"전체 고유 키워드 수: {len(df_freq)}개")
print(f"전체 키워드 등장 횟수: {len(all_kws)}회")
display(df_freq.head(50))

In [ ]:
FREQ_THRESHOLD = 3  

df_low = df_freq[df_freq['빈도'] <= FREQ_THRESHOLD].sort_values('빈도')
print(f"빈도 {FREQ_THRESHOLD} 이하 키워드: {len(df_low)}개")
print(f"전체 고유 키워드 중 {len(df_low)/len(df_freq)*100:.1f}% 차지")
display(df_low.head(20))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top40 = df_freq.head(40)
axes[0].barh(top40['키워드'][::-1], top40['빈도'][::-1], color='steelblue')
axes[0].set_title('상위 40개 키워드 빈도', fontsize=13)
axes[0].set_xlabel('빈도')

axes[1].hist(df_freq['빈도'], bins=50, color='salmon', edgecolor='white')
axes[1].axvline(FREQ_THRESHOLD, color='red', linestyle='--', label=f'임계값 ({FREQ_THRESHOLD})')
axes[1].set_title('키워드 빈도 분포', fontsize=13)
axes[1].set_xlabel('빈도')
axes[1].set_ylabel('키워드 수')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 3. IP-제품 속성 키워드 정렬 및 브릿지 분석 (Keyword Alignment)

In [ ]:
IP_ATTR_DIR = os.path.join(BASE_DIR, "data", "processed", "IP_속성추출")
IP_PATH = os.path.join(IP_ATTR_DIR, "IP_속성.json")
PROD_PATH = os.path.join(IP_ATTR_DIR, "제품_속성.json")
VERIFIED_CSV = os.path.join(BASE_DIR, "data", "raw", "눈검증_끝난_데이터셋_최종후보.csv")  # raw → processed 이동 전 임시 경로

if os.path.exists(IP_PATH) and os.path.exists(PROD_PATH):
    with open(IP_PATH, 'r', encoding='utf-8') as f:
        ip_df = pd.DataFrame(json.load(f))
    with open(PROD_PATH, 'r', encoding='utf-8') as f:
        prod_df = pd.DataFrame(json.load(f))
    print(f"✅ LLM 데이터 로드 완료: IP({len(ip_df)}건), 제품({len(prod_df)}건)")
else:
    print("⚠️ LLM JSON 파일을 찾을 수 없습니다.")
    ip_df = pd.DataFrame()
    prod_df = pd.DataFrame()

if os.path.exists(VERIFIED_CSV):
    df_ver = pd.read_csv(VERIFIED_CSV, encoding='euc-kr')
    df_ver.columns = ['중분류', '상품명', '관련있음']
    def parse_ver_kws(val):
        if pd.isna(val) or str(val).strip() == '': return []
        return [w.strip() for w in str(val).split(',') if w.strip()]

    df_ver['keywords_list'] = df_ver['관련있음'].apply(parse_ver_kws)
    print(f"✅ 검증 데이터셋 로드 완료: {len(df_ver)}개 상품")
else:
    print(f"⚠️ 검증 데이터셋 파일을 찾을 수 없습니다: {VERIFIED_CSV}")
    df_ver = pd.DataFrame(columns=['상품명', 'keywords_list'])

In [ ]:
def get_flat_list(series):
    res = []
    for x in series.dropna():
        if isinstance(x, list): res.extend(x)
        else: res.append(str(x))
    return res

ip_kws_raw = get_flat_list(ip_df['signature_keywords']) if 'signature_keywords' in ip_df.columns else []
ip_kw_counts = Counter(ip_kws_raw)
ip_set = set(ip_kw_counts.keys())

prod_cols = ['flavor', 'texture', 'tpo']
prod_kws_raw = []
for col in prod_cols:
    if col in prod_df.columns:
        prod_kws_raw.extend(get_flat_list(prod_df[col]))
prod_kw_counts = Counter(prod_kws_raw)
prod_set = set(prod_kw_counts.keys())

ver_kws_raw = get_flat_list(df_ver['keywords_list'])
ver_kw_counts = Counter(ver_kws_raw)
ver_set = set(ver_kw_counts.keys())

print(f"🔹 IP 고유 키워드: {len(ip_set)}개")
print(f"🔹 제품 고유 키워드: {len(prod_set)}개")
print(f"🔹 검증 데이터셋 고유 키워드: {len(ver_set)}개")

### 3.1. 교집합 및 차집합 분석 (Intersection & Difference)
두 데이터셋을 연결하는 핵심 키워드(교집합)와 한 쪽에만 치우친 키워드(차집합)를 살펴봅니다.

In [ ]:
intersection = ip_set.intersection(prod_set)
only_ip = ip_set - prod_set
only_prod = prod_set - ip_set

print(f"🌉 교집합(공통/브릿지): {len(intersection)}개 (비율: {len(intersection)/len(ip_set|prod_set)*100:.1f}%)")
print(f"🚩 IP 전용 키워드: {len(only_ip)}개")
print(f"🍱 제품 전용 키워드: {len(only_prod)}개")

inter_df = pd.DataFrame([
    {'키워드': kw, 'IP_빈도': ip_kw_counts[kw], '제품_빈도': prod_kw_counts[kw], '총합': ip_kw_counts[kw] + prod_kw_counts[kw]}
    for kw in intersection
]).sort_values('총합', ascending=False)

print("\n🔥 [상위 공통 키워드 (브릿지 후보)]")
display(inter_df.head(20))

In [ ]:
def plot_diff_top(kw_counts, diff_set, title, color):
    diff_counts = {kw: kw_counts[kw] for kw in diff_set}
    top_diff = sorted(diff_counts.items(), key=lambda x: x[1], reverse=True)[:20]
    
    k, v = zip(*top_diff)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=list(v), y=list(k), color=color)
    plt.title(title)
    plt.show()

plot_diff_top(ip_kw_counts, only_ip, "IP에만 있는 상위 키워드 (정렬 필요 후보)", "salmon")
plot_diff_top(prod_kw_counts, only_prod, "제품에만 있는 상위 키워드 (정렬 필요 후보)", "skyblue")

## Step 3.5. 데이터 소스 통합 정합성 분석 (Total 4 Sources)
인스타그램, LLM(IP/제품), 그리고 검증 데이터셋의 키워드가 얼마나 일치하는지 분석합니다.
이 단계에서 겹치지 않는 단어들은 네트워크 상에서 고립되므로 통합(Unification)이 필수적입니다.

In [ ]:
insta_set = set(kw_freq.keys())
ip_set = set(ip_set)
prod_set = set(prod_set)
ver_set = set(ver_set)
all_sets = insta_set | ip_set | prod_set | ver_set

print(f"📊 [Source 1: Insta] 고유 키워드: {len(insta_set)}개")
print(f"📊 [Source 2: LLM-IP] 고유 키워드: {len(ip_set)}개")
print(f"📊 [Source 3: LLM-Prod] 고유 키워드: {len(prod_set)}개")
print(f"📊 [Source 4: Verified] 고유 키워드: {len(ver_set)}개")
print(f"🔥 [Total] 전체 고유 키워드: {len(all_sets)}개")

bridge_all = insta_set.intersection(ip_set, prod_set, ver_set)
print(f"🔗 모든 소스에 공통인 핵심 브릿지: {len(bridge_all)}개")

In [ ]:
def print_overlap(name_a, set_a, name_b, set_b):
    inter = set_a.intersection(set_b)
    print(f"🤝 {name_a} ↔ {name_b} 교집합: {len(inter)}개")

print_overlap("Insta", insta_set, "LLM-IP", ip_set)
print_overlap("Insta", insta_set, "LLM-Prod", prod_set)
print_overlap("Insta", insta_set, "Verified", ver_set)
print_overlap("LLM-Prod", prod_set, "Verified", ver_set)

## Step 3.51. 노이즈 키워드 제거 (Keyword Noise Removal)

In [ ]:
REMOVE_KWS = {
    "11가지", "188cm", "1988", "20가지과채", "20인치",
    "25매거진", "39가지 디저트", "3가지맛", "3단", "3종구매",
    "6월한정", "70%", "7월3주차", "8찬구성", "9월신상주간",
    "Armageddon", "Black Mamba", "CF퀸", "DIY", "EVENT",
    "HK", "I Believe", "K-리드오프", "MBTIT", "ONLYCU",
    "CUXBAR", "Ode to Love", "SSALBOBBY", "Supernova", "가가와",
    "가격파괴", "가공품", "가스", "겹겹이살아있는결", "관형 캔",
    "국내 인기 캐릭터", "국민의뢰", "굴린", "기분따라", "끝내기홈런",
    "노릅", "단맛약함", "달지않음", "다크 판타지", "데몬헌터스",
    "데몬헌팅", "라이트세이버", "마을 동물", "맛있는", "맛있음",
    "머거본", "모든 가능성", "몰래카메라", "묻고더블로", "미국워싱턴",
    "미래적인", "미래지향적", "반값택배", "배불러", "비린내",
    "비린내거의없음", "비밀스러운", "빨간 단추", "빨간맛", "뾰루퉁한 표정",
    "브레디", "사직구장", "생미쉘", "샤도네이", "서주",
    "선데", "섬 꾸미기", "성인용음료", "세가지맛", "세계관",
    "세븐어클락", "세은", "세포들", "수삼", "스낭",
    "스무스", "스페이스 오페라", "시의들릱", "식집사", "실용함",
    "심리적", "심리전", "아모스필러즈", "아이폰17/호텔뷔페추첨이벤트", "아저규닫",
    "아홉 꼬리", "언제어디서나", "얼터너티브", "에리스리톨", "영웅적",
    "오구 패밀리", "오복채", "오색찬란", "온더락", "유심",
    "육아물", "음습한", "이중식", "작은 얼굴", "잔잔한여운",
    "잔잔함", "잔향", "장대한", "장수캐릭터", "잼의 정성",
    "제다이", "조르쥐뒤뵈프", "중식 여신", "지아잔틴", "지퍼",
    "지하감옥", "짤방", "차(tea)안주", "차(tea)와함께", "차와함께",
    "차분함", "책깃", "책읽기", "체인소", "초크초크",
    "커피와함께", "코믹함", "쿨톤", "클럽사운드", "클렌징",
    "탁구 선수", "탐구적", "탐미적", "탐험", "태평함",
    "터프함", "토이즈", "톤업", "퇴폐미", "편집샵",
    "평화로운", "포스텔러까리나", "폭스클럽", "학구적", "함께",
    "행정안전부", "현실적", "현장감", "협업", "혼자",
    "화장품", "화학", "활동", "홈런", "휴머니즘",
    "휴양지", "진정성", "진지함", "필수품", "필수템",
    "혈압관리", "혈액순환", "혈행", "푸근함",
}

def remove_noise_kws(kw_list):
    if not isinstance(kw_list, list): return kw_list
    return [kw for kw in kw_list if kw not in REMOVE_KWS]

all_sets  -= REMOVE_KWS
insta_set -= REMOVE_KWS
ip_set    -= REMOVE_KWS
prod_set  -= REMOVE_KWS
ver_set   -= REMOVE_KWS

df_qual['확정키워드_최종'] = df_qual['확정키워드_최종'].apply(remove_noise_kws)
if not ip_df.empty:
    ip_df['signature_keywords'] = ip_df['signature_keywords'].apply(remove_noise_kws)
if not prod_df.empty:
    for col in ['flavor', 'texture', 'tpo']:
        if col in prod_df.columns:
            prod_df[col] = prod_df[col].apply(remove_noise_kws)
if not df_ver.empty:
    df_ver['keywords_list'] = df_ver['keywords_list'].apply(remove_noise_kws)

print(f"노이즈 키워드 {len(REMOVE_KWS)}개 제거 완료")
print(f"제거 후 전체 고유 키워드: {len(all_sets)}개")

## Step 3.52. 키워드 동의어 통합 (Synonym Unification)

In [ ]:
SYNONYM_MAP = {
    # K팝 / 아이돌
    "KPopDemonHunters": "K팝데몬헌터스", "데몬헌터스": "K팝데몬헌터스",
    # 가벼움
    "가벼운": "가벼움", "가벼운한끼": "가벼움", "가볍게": "가벼움",
    "간단한식사": "가벼움", "간단한한끼": "가벼움",
    # 간식/간편
    "간식용": "간식",
    "간편식": "간편", "간편식사": "간편", "간편한식사": "간편",
    "간편한한끼": "간편", "간편함": "간편",
    # 감성/강렬
    "감성적": "감성", "감성적인": "감성",
    "강렬한": "강렬", "강렬한 인상": "강렬", "강렬함": "강렬",
    # 겨울/견과류/고급/고소
    "겨울철": "겨울",
    "견과": "견과류",
    "고급스러움": "고급", "고소함": "고소",
    # 에너지
    "기력보충": "에너지", "기력회복": "에너지", "기운채워주는": "에너지",
    "피로개선": "에너지", "피로해소": "에너지", "피로회복": "에너지",
    "에너제틱": "에너지", "에너지드링크": "에너지", "에너지보충": "에너지",
    # KBO / MLB
    "기아": "KBO", "기아타이거즈": "KBO", "야구": "KBO",
    "야구장": "KBO", "자이언츠": "KBO", "이글스": "KBO",
    "샌디에이고": "MLB", "샌프란시스코": "MLB", "양키스": "MLB",
    # 식사/즉석/중독성
    "식사대용": "식사", "식사메뉴": "식사", "식사반찬": "식사", "식사안주": "식사",
    "한끼": "식사", "한끼식사": "식사", "한끼식사대용": "식사",
    "즉석밥": "즉석", "즉석식품": "즉석",
    "중독성강한": "중독성", "중독적인": "중독성",
    # 크림 계열
    "슈": "크림", "슈크레": "크림", "슈크림": "크림", "생크림": "크림",
    "크리미": "크림", "크리미한 거품": "크림", "크리미함": "크림",
    # 상큼/새콤
    "상큼함": "상큼", "새콤함": "새콤", "새큼": "새콤",
    # 짭조름
    "짭조름": "짭조름함", "짭짤": "짭조름함", "짭짤함": "짭조름함", "짭쪼롬": "짭조름함",
    # 쫀득/쫄깃
    "쫀~득": "쫀득함", "쫀득": "쫀득함", "쫀득쫀득": "쫀득함", "쫀덕케": "쫀득함",
    "쫀쫀": "쫄깃함", "쫄깃": "쫄깃함",
    # 카레/카스테라/카라멜/티니핑
    "카레향": "카레", "커리": "카레", "커리향": "카레",
    "카스텔라": "카스테라",
    "캐라멜": "카라멜", "캐러멜": "카라멜", "캬라멜": "카라멜",
    "캐치!티니핑": "티니핑", "캐치티니핑": "티니핑",
    # 콜라보/쿠앤크/크레페
    "콜라보레이션": "콜라보", "쿠키앤크림": "쿠앤크",
    "크레이프": "크레페",
    # 탄산
    "톡쏘는": "탄산", "톡톡": "탄산", "톡톡터지는맛": "탄산", "톡톡터지는식감": "탄산",
    # 트렌디/특별함
    "트렌디한": "트렌디", "트렌디함": "트렌디",
    "특별식": "특별함", "특별한경험": "특별함", "특별한날": "특별함", "특별한식사": "특별함",
    # 한정판매/해장/행복
    "한정판": "한정판매", "할매입맛": "할매니얼",
    "해장용": "해장", "행복함": "행복",
    # 초코
    "초콜릿": "초코",
    # 흑백요리사
    "흑백요리사 셰프": "흑백요리사",
    # 2차 추가
    "405": "베이크하우스 405",
    "감귤향": "감귤",
    "꼬소": "고소", "꼬숩맛": "고소",
    "달콘": "달콤",
    "딸기향": "딸기",
    "블루 아카이브": "블루아카이브",
    "포켓몬": "포켓몬스터",
    "시즜": "시즌한정",
    "찐하고": "진함",
    "건강관리": "건강",
    "셀럽/인플루언서": "인플루언서", "셀럽": "인플루언서",
    "닌텐도 피크민": "피크민",
    "이미영조리사": "급식대가",
    "스타셰프": "셰프",
    "스윗함": "달콤",
    "제과제빵": "베이커리",
    "통팥": "팥",
}

SPLIT_MAP = {
    "간장소스": ["간장", "소스"],
    "굴소스": ["굴", "소스"],
    "데리야끼소스": ["데리야끼", "소스"],
    "로제 소스": ["로제", "소스"],
    "마요소스": ["마요", "소스"],
    "불닭소스": ["불닭", "소스"],
    "치킨소스": ["치킨", "소스"],
    "곤약면": ["곤약", "면"],
    "볶음면": ["볶음", "면"],
    "볶음밥": ["볶음", "밥"],
    "비빔면": ["비빔", "면"],
    "비빔밥": ["비빔", "밥"],
    "미역국": ["미역", "국"],
    "딸기잼": ["딸기", "잼"],
    "말차크림": ["말차", "크림"],
    "초코크림": ["초코", "크림"],
    "화이트초코": ["화이트", "초코"],
    "화이트초콜릿": ["화이트", "초코"],
    "딸기크림": ["딸기", "크림"],
    "새콤달콤": ["새콤", "달콤"],
    "감자칩": ["감자", "칩"],
    "감자탕": ["감자", "탕"],
    "청양고추": ["청양", "고추"],
    "청양마요": ["청양", "마요"],
    "치킨마요": ["치킨", "마요"],
    "허니버터": ["허니", "버터"],
    "딸기우유": ["딸기", "우유"],
    "솔티카라멜": ["솔티", "카라멜"],
    "바닐라라떼": ["바닐라", "카페"],
    "두산 유니폼": ["KBO", "유니폼"],
    "산리오 / 키티": ["산리오", "키티"],
    "단팥": ["팥", "달콤"],
    "소보로빵": ["소보로", "빵"],
    "찹쌀떡": ["찹쌀", "떡"],
    "건강간식": ["건강", "간식"],
    "당류ZERO": ["제로", "슈거"],
    "칼로리ZERO": ["제로", "칼로리"],
}

def unify_keywords(kw_list):
    if not isinstance(kw_list, list): return kw_list
    result = []
    for kw in kw_list:
        if kw in SPLIT_MAP:
            result.extend(SPLIT_MAP[kw])
        else:
            result.append(SYNONYM_MAP.get(kw, kw))
    return list(dict.fromkeys(result))

df_qual["확정키워드_최종"] = df_qual["확정키워드_최종"].apply(unify_keywords)
if not ip_df.empty:
    ip_df["signature_keywords"] = ip_df["signature_keywords"].apply(unify_keywords)
if not prod_df.empty:
    for col in ["flavor", "texture", "tpo"]:
        if col in prod_df.columns:
            prod_df[col] = prod_df[col].apply(unify_keywords)
if not df_ver.empty:
    df_ver["keywords_list"] = df_ver["keywords_list"].apply(unify_keywords)

all_sets  = set(unify_keywords(list(all_sets)))
insta_set = set(unify_keywords(list(insta_set)))
ip_set    = set(unify_keywords(list(ip_set)))
prod_set  = set(unify_keywords(list(prod_set)))
ver_set   = set(unify_keywords(list(ver_set)))

print(f"동의어 통합 완료: SYNONYM_MAP {len(SYNONYM_MAP)}개 규칙")
print(f"통합 후 전체 고유 키워드: {len(all_sets)}개")

## Step 3.53. 고급 키워드 전처리 (Advanced Preprocessing Rules)
우선순위: 조건부 예외 → 명시적 분할 → 포함(Contains) 통폐합 → 1:1 치환/제거

In [ ]:
SANDWICH_PRODUCTS = {
    '가격파괴랩샌드팩', '가격파괴패밀리샌드팩', '감자치즈베이컨샌드',
    '게맛있는샌드', '게살샐러드와 감자샐러드 샌드위치', '계란샌드베이글',
}

CONTAINS_COLLAPSE = ['예약', '예능', '원두', '아사이', '안주', '어른', '얼큰']

EXTRA_SPLIT_MAP = {
    "아침간식":     ["아침", "간식"],
    "아침식사":     ["아침", "식사"],
    "아침식사대용": ["아침", "식사"],
    "건강음료":     ["건강", "음료"],
    "계절":         ["계절", "행사"],
    "계절 행사":    ["계절", "행사"],
    "중화면":       ["중화", "면"],
    "중화빵":       ["중화", "빵"],
    "맘모롤":       ["맘모스", "롤"],
    "맘모스빵":     ["맘모스", "빵"],
    "일본가정식":   ["일본", "가정식"],
    "요리간편":     ["요리", "간편"],
    "와인안주":     ["와인", "안주"],
    "와인하이볼":   ["와인", "하이볼"],
    "에그마요":     ["에그", "마요"],
    "에그샐러드":   ["에그", "샐러드"],
    "에그말이":     ["에그", "말이"],
    "여름간식":     ["여름", "간식"],
    "여름음료":     ["여름", "음료"],
    "양념소스":     ["양념", "소스"],
    "양념치킨":     ["양념", "치킨"],
    "양념치킨볼":   ["양념", "치킨"],
    "깊은육수맛":   ["깊은맛", "육수"],
    "깊은육향":     ["깊은맛", "육향"],
    "깊은풍미":     ["깊은맛", "풍미"],
    "쌀가루":       ["쌀", "가루"],
    "쌀과자":       ["쌀", "과자"],
    "쌀떡":         ["쌀", "떡"],
    "쌀밥":         ["쌀", "밥"],
}

EXTRA_SYNONYM_MAP = {
    "건강식": "건강",    "건강함": "건강",
    "깊이": "깊은맛",    "깊이감": "깊은맛",
    "중화식": "중화",    "중화요리": "중화",    "중화풍": "중화",
    "녹는": "녹음",      "녹는식": "녹음",      "사르르": "녹음",
    "일본인 가옥거리": "일본",
    "요리재료": "요리",  "요리활용": "요리",
    "우리술": "술",      "우리쌀": "쌀",
    "여름철": "여름",
    "아침대용": "아침",
    "훗카이도": "홋카이도", "후카이도": "홋카이도",
    "은은한과일향": "과일", "은은한꽃향": "꽃",
    "은은한꿀향": "꿀",  "은은한단맛": "달콤",  "은은한풍미": "풍미",
}
REMOVE_WORDS = {"은은한향"}

def apply_advanced_rules(kw_list, product_name=""):
    if not isinstance(kw_list, list):
        return kw_list
    result = []
    for kw in kw_list:
        if kw == '제로':
            if product_name == '제로모히또제로카페인':
                result.extend(['제로', '카페인'])
            else:
                result.extend(['제로', '슈거'])
            continue
        if kw == '샌드':
            result.append('샌드위치' if product_name in SANDWICH_PRODUCTS else '샌드')
            continue
        if kw in EXTRA_SPLIT_MAP:
            result.extend(EXTRA_SPLIT_MAP[kw])
            continue
        collapsed = next((w for w in CONTAINS_COLLAPSE if w in kw), None)
        if collapsed:
            result.append(collapsed)
            continue
        if kw in REMOVE_WORDS:
            continue
        result.append(EXTRA_SYNONYM_MAP.get(kw, kw))
    return list(dict.fromkeys(result))

df_qual["확정키워드_최종"] = df_qual.apply(
    lambda row: apply_advanced_rules(row["확정키워드_최종"], row.get("제품명", "")),
    axis=1
)
if not ip_df.empty:
    ip_df["signature_keywords"] = ip_df["signature_keywords"].apply(apply_advanced_rules)
if not prod_df.empty:
    for col in ["flavor", "texture", "tpo"]:
        if col in prod_df.columns:
            prod_df[col] = prod_df[col].apply(apply_advanced_rules)
if not df_ver.empty:
    df_ver["keywords_list"] = df_ver["keywords_list"].apply(apply_advanced_rules)

all_sets  = set(apply_advanced_rules(list(all_sets)))
insta_set = set(apply_advanced_rules(list(insta_set)))
ip_set    = set(apply_advanced_rules(list(ip_set)))
prod_set  = set(apply_advanced_rules(list(prod_set)))
ver_set   = set(apply_advanced_rules(list(ver_set)))

print(f"고급 전처리 완료 — 전체 고유 키워드: {len(all_sets)}개")

## Step 3.6. 동의어 발굴 및 단어 분석 (Keyword Discovery & Drill-down)

In [ ]:
discovery_data = []
for kw in sorted(list(all_sets)):
    discovery_data.append({
        "키워드": kw,
        "Insta": kw_freq.get(kw, 0),
        "LLM_IP": ip_kw_counts.get(kw, 0),
        "LLM_Prod": prod_kw_counts.get(kw, 0),
        "Verified": ver_kw_counts.get(kw, 0),
        "총합": kw_freq.get(kw, 0) + ip_kw_counts.get(kw, 0) + prod_kw_counts.get(kw, 0) + ver_kw_counts.get(kw, 0)
    })

df_discovery = pd.DataFrame(discovery_data).sort_values("총합", ascending=False)
DISCOVERY_CSV = os.path.join(BASE_DIR, "eda", "keyword_discovery_full_list.csv")
df_discovery.to_csv(DISCOVERY_CSV, index=False, encoding="utf-8-sig")

print(f"✅ 전체 키워드 리스트({len(df_discovery)}건) CSV 저장 완료: {DISCOVERY_CSV}")
display(df_discovery.head(50))

## Step 3.7. 키워드 통합 및 대표어 설정 (Keyword Unification)

In [ ]:
# ✅ 이 셀은 Step 3.52로 이전됨 (SYNONYM_MAP 전체 규칙은 위 Step 3.52 셀 참고)
# Step 3.7은 더 이상 실행하지 않아도 됩니다.
print('Step 3.7: 동의어 통합은 Step 3.52에서 이미 처리되었습니다.')

In [ ]:
def search_keyword_origin(query):
    """4개 소스 전체에서 특정 단어가 포함된 제품/게시물과 키워드 맥락을 상세 검색"""
    print(f"🔍 '{query}' 키워드 포함 데이터 상세 검색 결과\n" + "="*60)
    
    insta_hits = df_qual[
        df_qual['제품명'].str.contains(query, na=False) | 
        df_qual['확정키워드_최종'].apply(lambda x: any(query in str(k) for k in x) if isinstance(x, list) else False)
    ]
    if not insta_hits.empty:
        print(f"\n[Source 1: Instagram] {len(insta_hits)}건 발견")
        display(insta_hits[['제품명', '확정키워드_최종']].head(15))
        
    if not ip_df.empty:
        ip_hits = ip_df[
            ip_df['keyword'].str.contains(query, na=False) |
            ip_df['signature_keywords'].apply(lambda x: any(query in str(k) for k in x) if isinstance(x, list) else False)
        ]
        if not ip_hits.empty:
            print(f"\n[Source 2: LLM-IP] {len(ip_hits)}건 발견")
            display(ip_hits[['keyword', 'signature_keywords']].head(15))
            
    if not prod_df.empty:
        prod_hits = prod_df[
            prod_df['keyword'].str.contains(query, na=False) |
            prod_df.apply(lambda row: any(query in str(val) for col in ['flavor', 'texture', 'tpo'] if col in prod_df.columns for val in (row[col] if isinstance(row[col], list) else [row[col]])), axis=1)
        ]
        if not prod_hits.empty:
            print(f"\n[Source 3: LLM-Product] {len(prod_hits)}건 발견")
            display(prod_hits[['keyword', 'flavor', 'texture']].head(15))
            
    if not df_ver.empty:
        ver_hits = df_ver[
            df_ver['상품명'].str.contains(query, na=False) |
            df_ver['keywords_list'].apply(lambda x: any(query in str(k) for k in x) if isinstance(x, list) else False)
        ]
        if not ver_hits.empty:
            print(f"\n[Source 4: Verified] {len(ver_hits)}건 발견")
            display(ver_hits[['상품명', 'keywords_list']].head(15))

## Step 4. 최종 네트워크용 데이터 내보내기

In [ ]:
edge_data = []
for idx, row in df_qual.iterrows():
    product = row['제품명']
    kws = row['확정키워드_최종']
    if isinstance(kws, list):
        for kw in kws:
            edge_data.append({'product': product, 'keyword': kw, 'brand': row['brand']})

df_edges = pd.DataFrame(edge_data)
EXPORT_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'network_edges_instagram.csv')
df_edges.to_csv(EXPORT_PATH, index=False, encoding='utf-8-sig')

print(f"✅ 네트워크 엣지 데이터 저장 완료: {EXPORT_PATH}")
print(f"총 엣지 수: {len(df_edges)}개")